In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
"""
preprocessing.py
-----------------
Carga y preprocesamiento de los flujos de red (CICFlowMeter) utilizados para
el pipeline de Threat Hunting Autonomo basado en Deep Learning.

Pasos:
1. Carga del CSV.
2. Limpieza: eliminacion de 'Timestamp' (ventana de captura de ~2h15min,
   sin variabilidad ciclica aprovechable) y de columnas de varianza cero
   (no aportan senal, p.ej. banderas TCP que nunca cambian en esta muestra).
3. Codificacion del target: 1 = any attack (clase de interes / amenaza
   a detectar para disparar el playbook de respuesta), 0 = Benign.
4. Particion estratificada train/val/test (60/20/20), fija para las 4
   semillas de cada modelo (comparacion justa: solo cambia la
   inicializacion/orden de mini-batches, no el split).
5. Estandarizacion (StandardScaler) ajustada solo con el set de train.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

TARGET_COL = "Label"

def load_raw(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    return df


LEAKAGE_COLS = ["Dst Port", "Protocol"]
# 'Dst Port' separa PERFECTAMENTE las clases en esta captura, por ejemplo
# (FTP-BruteForce = puerto 21 en el 100% de los casos; Benign = nunca
# puerto 21). Usarlo produce Accuracy/F1 = 1.0 en cualquier modelo, pero
# es "port fingerprinting", no deteccion de comportamiento -- un
# hallazgo bien documentado en la literatura de IDS con ML (ver p.ej.
# Engelen et al., 2021, sobre fugas de informacion en CICIDS). Se excluye
# por defecto para forzar una evaluacion conductual genuina, relevante
# para un modelo de threat hunting que debe generalizar a puertos no
# vistos.


def clean(df: pd.DataFrame, drop_leakage: bool = True) -> pd.DataFrame:
    df = df.copy()

    if "Timestamp" in df.columns:
        df = df.drop(columns=["Timestamp"])

    if drop_leakage:
        cols_to_drop = [c for c in LEAKAGE_COLS if c in df.columns]
        df = df.drop(columns=cols_to_drop)

    numeric_cols = df.select_dtypes(include="number").columns
    zero_var_cols = [c for c in numeric_cols if df[c].nunique(dropna=True) <= 1]
    if zero_var_cols:
        df = df.drop(columns=zero_var_cols)

    # Seguridad: reemplaza posibles inf/-inf (division por duracion ~0)
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna()

    return df, zero_var_cols


def encode_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Label"] = df["Label"].astype(str).str.strip()

    df["y"] = (df["Label"] != "Benign").astype(int)

    df = df.drop(columns=["Label"])
    return df


def split_and_scale(df: pd.DataFrame, random_state: int = 42):
    """Split estratificado unico (60/20/20) + escalado con train."""
    y = df["y"].values
    X = df.drop(columns=["y"]).values
    feature_names = df.drop(columns=["y"]).columns.tolist()

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.40, stratify=y, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=random_state
    )

    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_test": X_test, "y_test": y_test,
        "feature_names": feature_names,
        "scaler": scaler,
    }


def prepare_dataset(csv_path: str, random_state: int = 42, drop_leakage: bool = True):
    df = load_raw(csv_path)

    # Guardar resumen de labels originales antes de codificar
    df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()
    original_label_counts = df[TARGET_COL].value_counts().to_dict()
    detected_attack_labels = sorted(
        label for label in original_label_counts.keys()
        if label != "Benign"
    )

    df, dropped_cols = clean(df, drop_leakage=drop_leakage)
    df = encode_target(df)
    data = split_and_scale(df, random_state=random_state)
    data["dropped_zero_var_cols"] = dropped_cols
    data["n_features"] = data["X_train"].shape[1]
    data["class_counts_train"] = pd.Series(data["y_train"]).value_counts().to_dict()
    data["original_label_counts"] = original_label_counts
    data["detected_attack_labels"] = detected_attack_labels
    data["target_mapping"] = {
        0: "Benign",
        1: "Attack / Non-Benign"
    }
    return data


if __name__ == "__main__":
    d = prepare_dataset("/content/drive/MyDrive/02-15-2018.csv")
    print("N features:", d["n_features"])
    print(
        "Train shape:", d["X_train"].shape,
        "Val:", d["X_val"].shape,
        "Test:", d["X_test"].shape
    )
    print("Mapeo del target:", d["target_mapping"])
    print("Distribucion original de labels:")
    for label, count in d["original_label_counts"].items():
        print(f"  {label}: {count}")
    print("Labels considerados como ataque:")
    for label in d["detected_attack_labels"]:
        print(f"  - {label}")
    print("Distribucion train codificada (0=Benign, 1=Ataque):", d["class_counts_train"])
    print("Columnas varianza cero descartadas:", d["dropped_zero_var_cols"])


N features: 66
Train shape: (624328, 66) Val: (208110, 66) Test: (208110, 66)
Mapeo del target: {0: 'Benign', 1: 'Attack / Non-Benign'}
Distribucion original de labels:
  Benign: 996077
  DoS attacks-GoldenEye: 41508
  DoS attacks-Slowloris: 10990
Labels considerados como ataque:
  - DoS attacks-GoldenEye
  - DoS attacks-Slowloris
Distribucion train codificada (0=Benign, 1=Ataque): {0: 592829, 1: 31499}
Columnas varianza cero descartadas: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg']
